In [3]:
# Data download
!mkdir -p ../../data/others
!curl -o ../../data/others/sgjp-20261018.tab.gz https://download.sgjp.pl/morfeusz/20261018/sgjp-20261018.tab.gz
!gunzip -f ../../data/others/sgjp-20261018.tab.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 40.7M  100 40.7M    0     0  2340k      0  0:00:17  0:00:17 --:--:-- 1972k


# Normalizacja tekstu

Dzisiaj zajmiemy się pracą z "surowym" tekstem zawierającym szereg zanieczyszczeń których będziemy się chcieli pozbyć.

W pierwszym kroku zapoznamy się jednak z zasobem jakim jest Słownik Gramatyczny Języka Polskiego (https://sgjp.pl/leksemy/), a w szczególności jego udostępniana publicznie reprezentacja w postaci kolumnowej, dostępna w ramach narzędzia Morfeusz (https://morfeusz.sgjp.pl/).

Reprezentacja kolumnowa oznacza podobną organizację danych do tej którą znamy już z formatów **conllu** lub **csv**. W tym wypadku separatorem jest znak tabulacji: `\t`. Możemy więc skorzystać z biblioteki `pandas` do wczytania słownika jako tabeli. Wiersze oznaczają kolejno: 
1. formę tekstową
2. lemat - formę słownikową
3. znacznik morfosyntaktyczny
4. oznaczenia nazw własnych
5. dodatkowe kwalifikatory

In [4]:
import pandas
from pathlib import Path

DATA_DIR = Path("../../data")

dict_path = DATA_DIR / "others" / "sgjp-20261018.tab"
dictionary = pandas.read_csv(dict_path, sep="\t", skiprows=list(range(28)), header=None, names=("form", "lemma", "tag", "ner", "qual"))
dictionary

,form,lemma,tag,ner,qual
0,AA,AA:Sm1_pt,subst:pl:nom.gen.dat.acc.inst.loc.voc:m1:pt,nazwa_organizacji,NaN
1,AA,AA:Sm3.n_ncol,subst:sg.pl:nom.gen.dat.acc.inst.loc.voc:m3,nazwa_firmy,NaN
2,AA,AA:Sm3.n_ncol,subst:sg.pl:nom.gen.dat.acc.inst.loc.voc:n:ncol,nazwa_firmy,NaN
3,AAP,AAP,subst:sg.pl:nom.gen.dat.acc.inst.loc.voc:f,nazwa_instytucji,NaN
4,ABB,ABB,subst:sg.pl:nom.gen.dat.acc.inst.loc.voc:n:ncol,nazwa_firmy,NaN
...,...,...,...,...,...
7458538,asp,aspirant,brev:pun,NaN,NaN
7458539,sierż,sierżant,brev:pun,NaN,NaN
7458540,kom,komendant,brev:pun,NaN,NaN
7458541,bryg,brygada,brev:pun,NaN,NaN


Pierwszym krokiem który wykonamy, będzie redukcja słownika do zbioru unikalnych form tekstowych:

In [7]:
vocab = set(dictionary["form"])
print(f"{len(vocab)} unique forms")

5028873 unique forms


Spójrzmy więc na materiał, z którym będziemy dzisiaj pracowali: jest to tekst powieści Lalka Bolesława Prusa, uzyskany poprzez procedurę *Optical Character Recognition* (OCR) na podstawie PDFa udostępnionego przez serwis https://wolnelektury.pl

In [8]:
def read_file(fpath): # helper function
    with open(fpath) as f:
        txt = f.read()
    return txt

raw_lalka = read_file(DATA_DIR / "lektury" / "lalka.txt")
print(raw_lalka[:7000])



<!-- Start of picture text -->
4 al<br>. Lead a!<br>LE RS Se<br>| See— “> a oe et) ot<br>ay fags aes jas, er i “Saar:<br>. —¥ € = ad |<br>;<br>aa eh<br><!-- End of picture text -->

Ta lektura, podobnie �ak tysiące innych, �est dostępna on-line na stronie NI wolnelektury.pl , podobnie . jak tysigce innych, jest dostepna on-line na stronie OM Utwór opracowany został w ramach pro�ektu Utwér opracowany zostal w ramach projektu Wolne Lektury przez fundac�ę Wolne Lektury. <u>wolnelektury.pl</u> dacjg Wolne Lektury 

#### BOLESŁAW PRUS 

# _Lalka_ 

# _Lalka_ 

## TOM I 

### I. J�J. MS. W � 

W początkach roku , kiedy świat polityczny za�mował się poko�em san-stefańskim¹, wyborem nowego papieża² albo szansami europe�skie�wo�ny³, warszawscy kupcy tuǳież inteligenc�a pewne�okolicy Krakowskiego Przedmieścia⁴ niemnie�gorąco interesowała się przyszłością galantery�nego sklepu pod firmą J. Mincel i S. Wokulski. 

W renomowane��adłoda�ni, gǳie na wi

Jak widzimy, jakość tekstu pozostawia wiele do życzenia. Poznamy dzisiaj kilka technik, które są pomocne przy przygotowaniu - preprocessingu tekstów na poczet późniejszych prac. Najprawdopodobniej nie uda nam się usunąć wszystkich zanieczyszczeń, ale dobrym podejściem jest sukcesywne eliminowanie tych, które najbardziej rzucają się w oczy przez ich częstość. Podzielmy więc tekst prostym tokenizatorem, i wygenerujmy listę frekwenycjną form występujących w tekście:

In [10]:
import re

tokenizer = re.compile(r"\b\S+\b")
tokens = tokenizer.findall(raw_lalka)
df = pandas.DataFrame({"token": tokens})
df["token"].value_counts().to_frame().iloc[0:30]

,count
token,
się,6928
i,6737
w,4832
nie,4466
na,4180
z,3902
że,3234
do,3152
a,2754


Pierwsza obserwacja:

Do tekstu powieści dostały się elementy metatekstowe - przypisy redakcyjne:

In [11]:
footnote_index = raw_lalka.index("redakcy�ny")
margin = 400
print(raw_lalka[footnote_index-margin:footnote_index+margin])

usiała uznać niepodległość swych dotychczasowych państw lenniczych: Rumunii, Serbii i Czarnogórza, przy czym granice tych dwóch ostatnich zostały znacznie rozszerzone. Jednocześnie utworzono rozległe Księstwo Bułgarskie z dostępem do Morza Ege�skiego (obe�mu�ące Macedonię). Poza tym Turc�a została zmuszona do ustępstw terytorialnych na rzecz Ros�i w Az�i i do zapłaty wysokie�kontrybuc�i. [przypis redakcy�ny] 

² _wybór nowego papieża_ — lutego r., po śmierci Piusa IX, został papieżem kardynał Joachim Pecci �ako Leon XIII, znany późnie�ze swych wystąpień przeciw ruchowi soc�alistycznemu. [przypis redakcy�ny] 

³ _szanse europejskiej wojny_ — zwycięstwa Ros�i w wo�nie z Turc�ą, możliwość za�ęcia Konstantynopola przez wo�ska rosy�skie, a potem pokó�w San Stefano wywołały silne niezadowo


Widzimy, że zachowują się one dość regularnie, zdefiniujmy wyrażenie regex pozwalający na wykrycie ich w tekście:

In [12]:
footnote_re = re.compile(r"^.+redakcy�ny\] $", flags=re.MULTILINE)
footnote_re.findall(raw_lalka)

['¹ _pokój san-stefański_ — traktat zawarty mięǳy Ros�ą a Turc�ą dn. \uf733marca \uf731\uf738\uf737\uf738r. w San Stefano (pod Konstantynopolem). Turc�a, pokonana uprzednio w wo�nie z Ros�ą (\uf731\uf738\uf737\uf737–\uf731\uf738\uf737\uf738), musiała uznać niepodległość swych dotychczasowych państw lenniczych: Rumunii, Serbii i Czarnogórza, przy czym granice tych dwóch ostatnich zostały znacznie rozszerzone. Jednocześnie utworzono rozległe Księstwo Bułgarskie z dostępem do Morza Ege�skiego (obe�mu�ące Macedonię). Poza tym Turc�a została zmuszona do ustępstw terytorialnych na rzecz Ros�i w Az�i i do zapłaty wysokie�kontrybuc�i. [przypis redakcy�ny] ',
 '² _wybór nowego papieża_ — \uf732\uf730lutego \uf731\uf738\uf737\uf738r., po śmierci Piusa IX, został papieżem kardynał Joachim Pecci �ako Leon XIII, znany późnie�ze swych wystąpień przeciw ruchowi soc�alistycznemu. [przypis redakcy�ny] ',
 '³ _szanse europejskiej wojny_ — zwycięstwa Ros�i w wo�nie z Turc�ą, możliwość za�ęcia Konstantyno

Możemy usunąć te wszystkie wystąpienia metodą `sub` obiektu `re.Pattern`:

In [13]:
clean_lalka = footnote_re.sub("", raw_lalka)
print(f"Reduction: {len(clean_lalka)}/{len(raw_lalka)}", "\n")
print(clean_lalka[footnote_index-margin:footnote_index+margin])

Reduction: 1666295/1797631 

a to gruby interes — wtrącił pan Szprot, a�ent handlowy¹¹, który bywał tu rzadkim gościem. 

— Nic nie zrobi — odparł pan Deklewski — a tymczasem porządny sklep diabli wezmą. Na dostawach bogacą się tylko Żyǳi i Niemcy; nasi do tego nie ma�ą głowy. 

- A może Wokulski ma głowę? 

- Wariat! wariat!… — mruknął radca. — Poda�no, Józiu, piwa. Która to?… 

Alkohol, Pĳaństwo 

- Siódma buteleczka, panie radco. Służę piorunem. 

- Już siódma?… Jak ten czas leci, �ak ten czas leci… 

A�ent handlowy, który z obowiązków stanowiska potrzebował mieć o kupcach wiadomości wszechstronne i wyczerpu�ące, przeniósł swo�ą butelkę i szklankę do stołu radcy i topiąc słodkie we�rzenie w �ego załzawionych oczach, spytał zniżonym głosem: 

— Przepraszam, ale… Dlaczego pan radca nazywa Wokulskiego wariatem?… Może 


Druga obserwacja:

Teksty wydawane przez Wolne Lektury są często anotowane motywami reprezentowanymi w tekście, np. tutaj mamy

`Alkohol, Pĳaństwo`

Najczęśćiej są one zapisane jako przedzielone przecinkami frazy występujące w osobnej linii. Zobaczmy listę najczęściej występujących w powieści linii:

In [14]:
lines = clean_lalka.split("\n")
line_df = pandas.DataFrame({"line": lines})
line_counts = line_df.value_counts()
line_counts.to_frame().iloc[:50]

,count
line,
,14137
B�P _Lalka_,454
Sługa,34
Kobieta,28
Handel,20
Żyd,19
Miłość,19
Pozyc�a społeczna,18
Pieniąǳ,15


Widzimy kilka godnych odnotowania wyjątków, poza którymi niemal wszystkie z wypisanych linii stanowią element obcy w powieści. Usuńmy je:

In [15]:
exceptions = [
    "",
    "— Tak. ",
    "Wokulski zamyślił się. ",
    "Wokulski uśmiechnął się. "  
]

for motif_phrase in line_counts.index[:50]:
    motif_phrase = motif_phrase[0]
    if motif_phrase not in exceptions:
        motif_re = re.compile(r"^" + re.escape(motif_phrase) + r"$", flags=re.MULTILINE)
        clean_lalka = motif_re.sub("", clean_lalka)

print(f"Reduction: {len(clean_lalka)}/{len(raw_lalka)}", "\n")

Reduction: 1653370/1797631 



## Ćwiczenie 1
Jak widzieliśmy linie sygnalizujące motywy są często przedzieloną przecinkami konkatenacją indywidualnych motywów. Zaproponuj wyrażenie regularne które wychwyci takie linie na podstawie wyliczenia indywidualnych motywów i sposobu ich połączenia w linię, jakiej przykłady obserwujemy w tekście

In [21]:
# TODO
mortif_list = [m[0] for m in line_counts.to_frame().iloc[:50].index.tolist() if m[0] not in exceptions]
print(mortif_list)

#motifs_alternative = "|".join(motif_list)

#solution = re.compile(r"^")



#solution = re.compile(r"^(\w+, )+(\w+) $", flags = re.MULTILINE)
#for match in solution.finditer(clean_lalka):
   # print(match.group(0))

['B\uf76f\uf76c\uf765\uf773�\uf761\uf777P\uf772\uf775\uf773 _Lalka_ ', 'Sługa ', 'Kobieta ', 'Handel ', 'Żyd ', 'Miłość ', 'Pozyc�a społeczna ', 'Pieniąǳ ', 'Warszawa ', 'Podróż ', 'Praca ', 'Miasto ', 'Małżeństwo ', 'Dom ', 'Samobó�stwo ', 'Pies ', 'Koń ', 'Flirt ', 'Sen ', 'Choroba ', 'Jeǳenie ', 'List ', 'Ta�emnica ', 'Kobieta, Mężczyzna ', 'Łzy ', 'Teatr ', 'Nauka ', 'Modlitwa ', 'ǲiecko ', 'Salon ', 'Kondyc�a luǳka ', 'Starość ', 'Śmiech ', 'Zazdrość ', 'Marzenie ', 'Theatrum mundi ', 'Miasto, Warszawa ', 'Podstęp ', 'Nuda ', 'Obraz świata ', 'Szlachcic ', 'Jeǳenie, Obycza�e ', 'Swaty ', '— Nie. ', 'Odwaga ', 'Wo�na ']


Wykorzystajmy teraz słownik, do identyfikacji tokenów, które do niego nie pasują - tokeny takie nazywamy **Out of Vocabulary** (OOV):

In [22]:
clean_tokens = tokenizer.findall(clean_lalka)
clean_lalka_df = pandas.DataFrame({"token": clean_tokens})
clean_lalka_df["oov"] = ~(clean_lalka_df["token"].isin(vocab))
print(clean_lalka_df["oov"].value_counts(normalize=True))
clean_lalka_df

oov
False    0.76524
True     0.23476
Name: proportion, dtype: float64


,token,oov
0,Start,True
1,of,True
2,picture,True
3,text,True
4,4,True
...,...,...
244492,konto,False
244493,szczegóły,False
244494,na,False
244495,stronie,False


Ponownie, interesują nas najczęściej występujące tokeny tego typu:

In [23]:
oov = clean_lalka_df[clean_lalka_df["oov"]]
oov.value_counts().iloc[:50]

token        oov 
się          True    6745
ak           True    1135
est          True    1128
uż           True     858
Nie          True     627
eszcze       True     549
barǳo        True     436
ą            True     390
ǳiś          True     337
eżeli        True     321
To           True     311
No           True     287
ednak        True     282
Na           True     278
bęǳie        True     270
siebie       True     267
O            True     266
Czy          True     260
Więc         True     228
gǳie         True     226
estem        True     224
Ja           True     216
akby         True     209
Tak          True     204
Cóż          True     199
Ach          True     188
te�chwili    True     183
mięǳy        True     178
Po           True     175
Z            True     170
ǳień         True     167
luǳi         True     165
Może         True     165
akiś         True     163
Bo           True     157
Niech        True     143
poko�u       True     141
Już          True   

Znak "j" został niepoprawnie rozpoznany na etapie OCR, i jest zapisywany jako "�". Poprawmy to, i powtórzmy analizę:

In [24]:
clean_lalka = clean_lalka.replace("�", "j")
clean_tokens = tokenizer.findall(clean_lalka)
clean_lalka_df = pandas.DataFrame({"token": clean_tokens})
clean_lalka_df["oov"] = ~(clean_lalka_df["token"].isin(vocab))
print(clean_lalka_df["oov"].value_counts(normalize=True))
oov = clean_lalka_df[clean_lalka_df["oov"]]
oov.value_counts().iloc[:50]

oov
False    0.831278
True     0.168722
Name: proportion, dtype: float64


token         oov 
się           True    6745
Nie           True     627
barǳo         True     436
ǳiś           True     337
To            True     311
No            True     287
Na            True     278
bęǳie         True     270
siebie        True     267
Czy           True     260
O             True     250
Więc          True     228
gǳie          True     226
Ja            True     216
Tak           True     204
Cóż           True     199
Ach           True     188
tejchwili     True     183
mięǳy         True     178
Po            True     175
Z             True     170
ǳień          True     167
luǳi          True     165
Może          True     165
Bo            True     157
Niech         True     143
Już           True     136
Za            True     127
odpowieǳiał   True     123
Pani          True     123
pieniąǳe      True     117
Szlangbaum    True     115
luǳie         True     104
Jeżeli        True     100
Kto           True      99
On            True      98
Jest     

## Ćwiczenie 2:

Na trzeciej pozycji listy słów spoza słownika dalej znajduje się słowo "barǳo", spróbuj zidentyfikować przyczynę, i wyeliminować ją

In [26]:
# TODO
clean_lalka = clean_lalka.replace("ǳ", "dz")
clean_tokens = tokenizer.findall(clean_lalka)
clean_lalka_df = pandas.DataFrame({"token": clean_tokens})
clean_lalka_df["oov"] = ~(clean_lalka_df["token"].isin(vocab))
print(clean_lalka_df["oov"].value_counts(normalize=True))
oov = clean_lalka_df[clean_lalka_df["oov"]]
oov.value_counts().iloc[:50]

oov
False    0.866554
True     0.133446
Name: proportion, dtype: float64


token        oov 
się          True    6745
Nie          True     627
To           True     311
No           True     287
Na           True     278
siebie       True     267
Czy          True     260
O            True     250
Więc         True     228
Ja           True     216
Tak          True     204
Cóż          True     199
Ach          True     188
tejchwili    True     183
Po           True     175
Z            True     170
Może         True     165
Bo           True     157
Niech        True     143
Już          True     136
Za           True     127
Pani         True     123
Szlangbaum   True     115
Jeżeli       True     100
Kto          True      99
On           True      98
Jest         True      97
Zresztą      True      95
Maruszewicz  True      89
Gdy          True      84
Bóg          True      84
Geist        True      82
Kobieta      True      79
Gdyby        True      78
Zdawało      True      76
Do           True      75
Tu           True      74
Ależ         True   

## Ćwiczenie 3:

Słownik zawiera klucze o określonej wielkości znaków - poza nazwami własnymi i akronimami są to w większości formy pisane minuskułą. Jak wpływa to na status OOV? Sprawdź jakie będą najczęstsze tokeny OOV po wzięciu pod uwagę tej komplikacji

In [ ]:
# TODO

# Lematyzacja

Często możemy chcieć uprościć część złożoności w zakresie form z jakimi mamy do czynienia. Procedura ta może przybierać różne formy.

Język angielski charakteryzuje się ubogą morfologią, w jego wypadku formy odmiany czasowników (przez czas gramatyczny), rzeczowników (przez liczbę) i przymiotników (przez stopień) są bardzo regularne, i polegają w większości wypadków na sufiksacji - dołączaniu odpowiednich końcowek do wyrazu. W takim przypadku efektywny będzie *stemming* czyli obcinanie form tekstowych do ich tematu fleksyjnego.

Język polski wygląda diametralnie odmiennie: paradygmaty odmiany rzeczowników, czasowników i przymiotników są bardzo złożone, i liczne. Często trudno jest wyodrębnić temat, który pozwalałby jednoznacznie zidentyfikować słowo. Dlatego też w języku polskim najczęściej stosuje się procedurę *lematyzacji* czyli sprowadzenia słów występujących w tekście do ich postaci słownikowej. Lematyzacja ma za zadanie redukcję złożoności *fleksyjnej* - i.e. związanej z gramatyką, a nie słowotwórczej Słownik SGJP pozwala na przygotowanie narzędzia mapującego formy tekstowe na lematy:

In [32]:
from tqdm import tqdm

lemmatizer_dict = {}
pairs = dictionary[["form", "lemma"]].itertuples()
for tup in tqdm(pairs):
    form, lemma = tup.form, tup.lemma
    if lemma != ":":
        lemma = lemma.split(":")[0]
    if form in lemmatizer_dict:
        lemmatizer_dict[form].add(lemma)
    else:
        lemmatizer_dict[form] = {lemma}
# equivalent, more elegant, but much slower:
#lemmatizer = dictionary.groupby("form").agg({"lemma": list})["lemma"]
#lemmatizer_dict = lemmatizer.to_dict()

7458543it [00:10, 737537.47it/s] 


In [28]:
for form in ["dzieciątkami", "widzielibyśmy", "najpiękniejszemu"]:
    print(f"{form} -> {lemmatizer_dict[form]}")

dzieciątkami -> {'dzieciątko'}
widzielibyśmy -> {'widzieć'}
najpiękniejszemu -> {'piękny'}


Zwróćmy uwagę, że mapowanie to przypisuje formom *listy* lematów, i.e. wartości takich może być więcej niż jedna. Jest tak dlatego, że formy tekstowe są często niejednoznaczne, i dopiero na podstawie kontekstu wiemy które ze słów słownikowych jest reprezentowane przez dany ciąg znaków. W ekstremalnych przypadkach, jedna forma może korespondować z lematami należącymi do innych części mowy:

In [29]:
for form in ["ci", "tony", "mai", "stoi"]:
    print(f"{form} -> {lemmatizer_dict[form]}")

ci -> {'ten', 'ty', 'ci'}
tony -> {'tona', 'ton'}
mai -> {'maja', 'maić'}
stoi -> {'stać:Vi', 'stoa'}


Procedurę lematyzacji słownikowej możemy uzupełnić etapem dezambiguacji, która powoli wybrać najbardziej prawdopodobną interpretację. Prosty algorytm realizujący takie zadanie wykorzystuje tagi morfosyntaktyczne (zakładając że tekst jest otagowany). Spośród lematów zgadzających się co do morfologii (m.in. części mowy) z naszym słowem, np.:

"Pokażę *ci* jak brzmią te dwa *tony*"

1. "ci" jest dopełnieniem bliższym "Pokażę", więc musi być rzeczownikiem w celowniku, to wskazuje na lemat "ty".
2. "dwa" sygnalizuje jeden z rodzajów męskich, zatem "tony" musi być postacią "ton"

Jeżeli po filtrowaniu morfosyntaktycznym dalej mamy więcej niż jeden wariant do wyboru, możemy, posługując się danymi frekwencyjnymi wybrać najczęściej reprezentowany lemat. W naszym podejściu nie mamy dostępu do żadnego z tych zasobów, więc zadowolimy się klasami lematów:

In [30]:
clean_lalka_df["lower_token"] = clean_lalka_df["token"].apply(lambda x: x.lower())
clean_lalka_df["lemmas"] = clean_lalka_df["lower_token"].map(lemmatizer_dict)
clean_lalka_df["num_lemmas"] = clean_lalka_df["lemmas"].apply(lambda x: len(x) if x == x else 0) # handling NAN values
clean_lalka_df

,token,oov,lower_token,lemmas,num_lemmas
0,Start,True,start,{start},1
1,of,True,of,NaN,0
2,picture,True,picture,NaN,0
3,text,True,text,NaN,0
4,4,True,4,NaN,0
...,...,...,...,...,...
244498,konto,False,konto,{konto},1
244499,szczegóły,False,szczegóły,{szczegół},1
244500,na,False,na,"{na:P, na:I}",2
244501,stronie,False,stronie,{strona},1


Zobaczmy najbardziej niejednoznaczne słowa:

In [31]:
clean_lalka_df.sort_values("num_lemmas", ascending=False).drop_duplicates("token").iloc[:20]

,token,oov,lower_token,lemmas,num_lemmas
220445,w,False,w,"{wyspa, w, wieś, wielki, wiek, wiersz}",6
108871,to,False,to,"{to:T, to:M, to:V, to:S, ten, to:C}",6
108870,Jak,False,jak,"{jak:D, jaka, jak:C, jak:M, jak:P, jak:S}",6
98171,To,True,to,"{to:T, to:M, to:V, to:S, ten, to:C}",6
15727,jak,False,jak,"{jak:D, jaka, jak:C, jak:M, jak:P, jak:S}",6
48158,W,False,w,"{wyspa, w, wieś, wielki, wiek, wiersz}",6
6159,niż,False,niż,"{niż:C, nizać, niżyć, niża, niż:S, niż:P}",6
187500,Kole,False,kole,"{koł, kolo, kole, kola, koło:S, kół}",6
159636,co,False,co,"{co:Pacc.nom, co:Pgen, co:S, co:T, co:M}",5
77310,Jako,True,jako,"{jako:M, jako:C, jaka, jako:P, jako:F}",5


## Ćwiczenie 4:

Porównaj w jednej tabeli najczęściej występujące formy tekstowe, oraz najczęściej reprezentowane lematy (możesz założyć że w każdym wypadku pierwszy wylistowany lemat jest poprawny).

In [ ]:
# TODO


## Ćwiczenie 5:

Zastanów się jakie jeszcze zanieczyszczenia widzisz w tekście Lalki, i jak - korzystając z poznanych narzędzi - możesz je wyeliminować. Sprawdź jak wpłynie to na długość oczyszczonego tekstu, oraz proporcję słów spoza słownika

In [ ]:
# TODO